# RQ2 — Shared Risk Context và định tuyến / Context conformance and routing

**Trương Minh Hoàng · MSSV 25410008**

Tôi kiểm tra cách chuẩn hóa đầu ra tín dụng và bất thường đồ thị mà vẫn giữ rõ miền,
ý nghĩa điểm số và nguồn. Ví dụ dùng bản ghi **tạo mẫu**, theo hợp đồng KLTN v2.1.
Các nguồn không được xem là cùng khách hàng và không tạo điểm tín dụng–gian lận chung.

Synthetic records exercise the actual thesis adapter and routing functions. They do not
establish cross-dataset entity linkage, predictive uplift or bank decision authority.

In [1]:
from pathlib import Path
import sys
PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if not (PROJECT / 'shared_risk_context').is_dir():
    raise FileNotFoundError('Run from this project directory or its notebooks directory.')
sys.path.insert(0, str(PROJECT))
import numpy as np
import pandas as pd
from IPython.display import display

In [2]:
from shared_risk_context import context
from shared_risk_context.examples import example_records, run_controlled_checks
records = example_records()
contexts = {key: context.adapt_record(record, key) for key, record in records.items()}
display(pd.DataFrame(contexts).T[['entity_scope', 'risk_domain', 'score_semantics',
                                  'score_value', 'assessment_time_status']])
for key, record in records.items():
    for field in context.SOURCE_REQUIRED:
        assert contexts[key][field] == record[field]
print('Source fields preserved in the synthetic examples: PASS')

,entity_scope,risk_domain,score_semantics,score_value,assessment_time_status
credit,lendingclub_loan_record,credit,lightgbm_default_class_score,0.18,source_observed_month
fraud_node_only,dgraphfin_user_node,fraud_graph,dgraphfin_anomaly_class_score,0.41,not_available_in_source
fraud_node_plus_graph,dgraphfin_user_node,fraud_graph,dgraphfin_anomaly_class_score,0.43,not_available_in_source


Source fields preserved in the synthetic examples: PASS


## 13 tình huống kiểm soát / Thirteen controlled scenarios

Tôi giữ các tình huống từ notebook KLTN: bản hợp lệ, thiếu trường, sai ngữ nghĩa,
sai phiên bản, điểm ngoài phạm vi, sai nguồn, thiếu tham chiếu và yêu cầu gộp điểm.
Each expected status and route is also checked against the historical scenario table.

In [3]:
result = run_controlled_checks()
cases = pd.DataFrame(result['controlled_cases'])
display(cases[['case_id', 'expected_status', 'status', 'expected_route', 'route', 'pass']])
assert result['controlled_cases_passed'] == result['controlled_case_count'] == 13

,case_id,expected_status,status,expected_route,route,pass
0,C01_VALID_CREDIT,ACCEPTED,ACCEPTED,CREDIT_REVIEW,CREDIT_REVIEW,True
1,C02_VALID_FRAUD_NODE,ACCEPTED,ACCEPTED,FRAUD_GRAPH_REVIEW,FRAUD_GRAPH_REVIEW,True
2,C03_VALID_FRAUD_GRAPH,ACCEPTED,ACCEPTED,FRAUD_GRAPH_REVIEW,FRAUD_GRAPH_REVIEW,True
3,C04_MISSING_MANDATORY,REJECTED,REJECTED,NONE,NONE,True
4,C05_WRONG_SEMANTICS,REJECTED,REJECTED,NONE,NONE,True
5,C06_UNSUPPORTED_MODEL,REJECTED,REJECTED,NONE,NONE,True
6,C07_SCORE_RANGE,REJECTED,REJECTED,NONE,NONE,True
7,C08_BAD_PROVENANCE,REJECTED,REJECTED,NONE,NONE,True
8,C09_REFERENCE_UNAVAILABLE,HELD,HELD,DATA_REVIEW,DATA_REVIEW,True
9,C10_JOINT_SCORE_FORBIDDEN,REJECTED,REJECTED,NONE,NONE,True


In [4]:
summary = {key: value for key, value in result.items() if key != 'controlled_cases'}
display(pd.Series(summary, name='current controlled checks'))
assert result['unique_output_hashes'] == 1 and result['determinism_repetitions'] == 100
assert result['order_invariant'] and result['multi_context_route'] == 'MULTI_CONTEXT_REVIEW'
assert result['entity_linkage_status'] == 'synthetic_not_observed'
assert result['joint_score'] is None
LAB_RESULT = {'scope': 'synthetic controlled scenarios using thesis v2.1 functions', **summary}
print(LAB_RESULT)

controlled_cases_passed                        13
controlled_case_count                          13
determinism_repetitions                       100
unique_output_hashes                            1
multi_context_route          MULTI_CONTEXT_REVIEW
order_invariant                              True
entity_linkage_status      synthetic_not_observed
joint_score                                  None
Name: current controlled checks, dtype: object

{'scope': 'synthetic controlled scenarios using thesis v2.1 functions', 'controlled_cases_passed': 13, 'controlled_case_count': 13, 'determinism_repetitions': 100, 'unique_output_hashes': 1, 'multi_context_route': 'MULTI_CONTEXT_REVIEW', 'order_invariant': True, 'entity_linkage_status': 'synthetic_not_observed', 'joint_score': None}


## Điều tôi rút ra / What I learned

Điểm số cùng nằm trong [0,1] chưa có nghĩa là có thể cộng hoặc so sánh trực tiếp.
Kiểm tra tính xác định xác nhận cùng đầu vào/chính sách cho cùng đầu ra trong phạm vi bài này;
nó chưa đánh giá chất lượng quyết định của con người hoặc hiệu quả vận hành.

Scores need semantics and provenance. Determinism and contract conformance are narrower
claims than decision quality. See the [project README](../README.md) for full-data replay.